In [5]:
from transformers import pipeline
from huggingface_hub import login
from getpass import getpass





In [9]:
token = getpass("Please enter your Hugging Face API token: ")

if token:
    login(token=token)
    del token


Please enter your Hugging Face API token: ··········


In [10]:
chatbot=pipeline(
    "text-generation",
    model="qwen/Qwen2.5-0.5B-Instruct",
    device="cpu"
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [11]:
messages=[
    {
        "role":"system",
        "cotent":(
            "you are a friend "
            "you are useful"
            "you are helping "
            "you are beutiful girlfriend "
        )
    }
]

In [13]:
# Fix the typo in the initial message 'cotent' -> 'content' if it exists
for msg in messages:
    if "cotent" in msg:
        msg["content"] = msg.pop("cotent")

while True:
    user_input = input("you: ")

    if user_input.lower() == "exit":
        break

    messages.append({
        "role": "user",
        "content": user_input
    })

    # Format the message history using the tokenizer's chat template
    prompt = chatbot.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Generate response
    response = chatbot(prompt, max_new_tokens=512, do_sample=True, temperature=0.7)

    # Extract and parse only the new response from the generated text
    generated_text = response[0]["generated_text"]
    assistant_response = generated_text[len(prompt):].strip()

    messages.append({
        "role": "assistant",
        "content": assistant_response
    })

    print("bot:", assistant_response)

you: hi
bot: Hello! How can I assist you today?
you: how can you help me 
bot: I'm here to help with any questions or tasks you might have. What would you like to know or do?
you: how to do good shopping 
bot: Shopping is an important part of life, and it's great that you want to make the most out of your time. Here are some tips on how to do good shopping:

1. Plan ahead: Before heading to the store, plan what you need. Make a list of items you want to buy and keep track of your budget.

2. Shop in person: Shopping in person allows you to see products up close and get a better sense of their quality and value.

3. Look for deals: Use coupons and discounts to save money on your purchases. Also, look for sales and promotions that are happening at stores.

4. Choose quality over quantity: Instead of buying everything you need, choose items that will last longer and add more value to your wallet.

5. Consider second-hand stores: Second-hand stores often offer unique and high-quality items

KeyboardInterrupt: Interrupted by user

In [15]:
%%bash
# Upgrade Streamlit to the latest version to resolve javascript asset loading errors
pip install -q --upgrade streamlit
npm install -q -g localtunnel


changed 22 packages in 843ms

3 packages are looking for funding
  run `npm fund` for details


In [19]:
%%writefile app.py
import streamlit as st
from transformers import pipeline
import torch

st.set_page_config(page_title="Simple Bot", layout="centered")
st.title("🤖 Super Light Chatbot")

@st.cache_resource
def load_model():
    # TinyLlama is under 700MB, loads extremely fast and works well on low memory
    return pipeline(
        "text-generation",
        model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        device=0 if torch.cuda.is_available() else -1,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
    )

chatbot = load_model()

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

if user_input := st.chat_input("Type your message..."):
    st.chat_message("user").write(user_input)
    st.session_state.messages.append({"role": "user", "content": user_input})

    # Prepare the simple conversation prompt template for TinyLlama
    prompt = f"<|system|>\nYou are a helpful coding assistant.</s>\n<|user|>\n{user_input}</s>\n<|assistant|>\n"

    with st.chat_message("assistant"):
        with st.spinner("..."):
            output = chatbot(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50)
            response_text = output[0]["generated_text"][len(prompt):].strip()
            st.write(response_text)

    st.session_state.messages.append({"role": "assistant", "content": response_text})

Overwriting app.py


In [17]:

import urllib
print("Your password for the localtunnel URL is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

Your password for the localtunnel URL is: 136.85.177.138


In [18]:
import subprocess
import sys
# Use sys.executable to run streamlit as a module directly with the current Python environment
subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501"])
!npx lt --port 8501

⠙your url is: https://short-teeth-repair.loca.lt
^C


In [20]:
# Install gradio for a zero-hassle, light UI inside the notebook
!pip install -q gradio

In [ ]:
import gradio as gr
from transformers import pipeline
import torch

# Using the ultra-lightweight DeepSeek-R1 1.5B distill model
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

chatbot_pipeline = pipeline(
    "text-generation",
    model=model_id,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

def respond(message, chat_history):
    # Format history to match standard chat assistant templates using list of dicts
    messages = []
    for msg in chat_history:
        messages.append({"role": msg["role"], "content": msg["content"]})
    messages.append({"role": "user", "content": message})

    formatted_prompt = chatbot_pipeline.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    outputs = chatbot_pipeline(
        formatted_prompt,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_k=50
    )

    response_text = outputs[0]["generated_text"][len(formatted_prompt):].strip()

    # Update chat history with new user message and assistant reply using dictionary format
    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": response_text})
    return "", chat_history

# Clean, ultra-light Gradio interface with standard dictionary-based history mapping
with gr.Blocks() as demo:
    gr.Markdown("## Light DeepSeek 1.5B Chatbot")
    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="Type your message here and press Enter...")
    clear = gr.Button("Clear Conversation")

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)

demo.launch(share=True, debug=True)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://94227c83d3f88b700e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7863 <> https://94227c83d3f88b700e.gradio.live
